# Week 3 — Theory
## Model diagnostics and evaluation visualization

A trained model is not done until you know **how it fails**. This notebook builds the
vocabulary and the chart catalogue you need for the lab:

1. The classifier-diagnostic family: **ROC**, **PR**, **calibration**, and the
   **confusion matrix**.
2. **Decision boundaries** — what they tell you and why we plot them.
3. The regressor-diagnostic family: **residuals**, **prediction-error plots**,
   **learning curves**.
4. The distinction between **discrimination** (can the model rank?) and **calibration**
   (do the probabilities mean what they say?).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons, make_regression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, mean_squared_error,
)
from sklearn.calibration import calibration_curve

plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(42)


## 1. ROC, PR, calibration — and when to use which

A binary classifier produces a continuous score. **The chart you pick depends on the
question you are asking of that score.**

| Chart        | Asks                                                  | Sensitive to imbalance? |
|--------------|-------------------------------------------------------|--------------------------|
| ROC          | "Can the model rank positives above negatives?"        | No                       |
| PR           | "Among predicted positives, how many are true?"        | **Yes**                  |
| Calibration  | "Does P(y=1) = 0.7 mean 70 % positives in practice?"   | Indirectly               |

### ROC curve

True positive rate (TPR) against false positive rate (FPR) as you sweep the decision
threshold from 1 down to 0. AUC = area under the curve = probability that a random
positive scores higher than a random negative. Insensitive to class imbalance, which is
both a strength (compare across datasets) and a weakness (does not tell you about
absolute error rates).

### PR curve

Precision against recall. **Use this when positives are rare** — ROC can look excellent
on a 1 %-positive dataset while precision at any usable threshold is still poor.

### Calibration curve

Bin predictions by predicted probability and plot mean predicted vs. observed positive
rate. A perfect classifier sits on the diagonal. Useful when you want to act on the
**probability** itself (e.g. decision-theoretic thresholding, risk reporting,
conformal prediction).

In [ ]:
# A balanced dataset to start with
X, y = make_classification(n_samples=2000, n_features=20, n_informative=8,
                           n_redundant=2, flip_y=0.05, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

# A reasonable nonlinear classifier
clf = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
scores = clf.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# ROC
fpr, tpr, _ = roc_curve(y_test, scores)
axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {auc(fpr, tpr):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC")
axes[0].legend(loc="lower right")

# PR
prec, rec, _ = precision_recall_curve(y_test, scores)
ap = average_precision_score(y_test, scores)
axes[1].plot(rec, prec, lw=2, label=f"AP = {ap:.3f}")
axes[1].set(xlabel="recall", ylabel="precision", title="Precision–Recall")
axes[1].legend(loc="lower left")

# Calibration
prob_pred, prob_true = calibration_curve(y_test, scores, n_bins=10, strategy="quantile")
axes[2].plot(prob_pred, prob_true, marker="o", lw=2)
axes[2].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[2].set(xlabel="mean predicted probability", ylabel="observed fraction positive",
            title="Calibration")
plt.tight_layout()
plt.show()


On a balanced, well-behaved dataset, all three charts agree the model is good.
The interesting cases are imbalanced ones — let's repeat with a 5 % positive rate.

In [ ]:
X, y = make_classification(n_samples=5000, n_features=20, n_informative=8,
                           weights=[0.95, 0.05], flip_y=0.02, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)
clf = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
scores = clf.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fpr, tpr, _ = roc_curve(y_test, scores)
axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {auc(fpr, tpr):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC — still looks great")
axes[0].legend(loc="lower right")

prec, rec, _ = precision_recall_curve(y_test, scores)
ap = average_precision_score(y_test, scores)
axes[1].plot(rec, prec, lw=2, label=f"AP = {ap:.3f}")
axes[1].axhline(y_test.mean(), color="grey", ls="--", lw=0.8,
                label=f"base rate = {y_test.mean():.3f}")
axes[1].set(xlabel="recall", ylabel="precision", title="PR — much more honest")
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.show()


The ROC AUC barely moves; the PR plot reveals that to get 80 % recall you have
to accept precision in the 0.4–0.5 range. **For rare-positive problems, lead with PR.**

The dashed line on the PR plot is the no-skill baseline — a classifier that picks the
positive class with probability equal to its prevalence. ROC's baseline is the y=x
diagonal; PR's baseline is the horizontal at the base rate.

## 2. Decision boundaries

A decision-boundary plot fixes the feature space to 2D, draws the model's predicted class
(or score) at every point, and overlays the training data. It is the most direct way to
**see** what the model has learned.

The trick: even for high-dimensional problems, we can run the diagnostic on the leading
two PCA components or on a UMAP projection. The resulting picture is approximate (the
boundary is in the original space), but the qualitative pattern — "is this a smooth
boundary or a step function?" — is preserved.

For the theoretical illustration we use a 2D toy problem so the boundary is exact:

In [ ]:
X, y = make_moons(n_samples=600, noise=0.25, random_state=0)
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 300), np.linspace(-1.0, 1.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name, m in zip(axes, ["LR", "GB shallow", "GB deep"],
                       [LogisticRegression(),
                        GradientBoostingClassifier(max_depth=2, n_estimators=20, random_state=0),
                        GradientBoostingClassifier(max_depth=5, n_estimators=400, random_state=0)]):
    m.fit(X, y)
    p = m.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=20, cmap="RdBu_r", alpha=0.7)
    ax.contour(xx, yy, p, levels=[0.5], colors="black", linewidths=1.2)
    ax.scatter(*X.T, c=y, cmap="RdBu_r", edgecolor="black", s=22, linewidth=0.4)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


Three pictures of three biases:

- **Logistic regression** can only learn a straight boundary. It will not solve moons.
- **GB shallow** learns a clean, almost-correct boundary that *generalises*.
- **GB deep** learns a wiggly boundary that hugs the training noise — classic
  overfitting. The same picture shows up in lab work as "validation accuracy plateaus
  while training accuracy climbs".

A boundary plot is worth more than three pages of validation-loss curves when you want
to communicate *what* a model is doing wrong.

## 3. Regression diagnostics

For regression we want to see whether errors are **unbiased** (residuals centred on
zero), **homoscedastic** (constant variance across predictions), and **uncorrelated with
the predicted value**. The two canonical charts:

- **Residuals vs. fitted**: residuals on y-axis, $\hat{y}$ on x-axis. A horizontal band
  of points is the goal. Funnel shapes signal heteroskedasticity; curves signal model
  misspecification.
- **Predicted vs. actual**: $\hat{y}$ on y-axis, $y$ on x-axis. Points should lie on
  the diagonal. Systematic departures from the diagonal indicate bias in some region.

In [ ]:
X, y = make_regression(n_samples=500, n_features=10, n_informative=5,
                       noise=10, random_state=0)
# Inject heteroskedasticity — error grows with |y|
y = y + RNG.normal(0, 1, size=y.shape) * np.abs(y) * 0.15

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)
reg = GradientBoostingRegressor(random_state=0).fit(X_train, y_train)
y_hat = reg.predict(X_test)
resid = y_test - y_hat

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(y_hat, resid, alpha=0.6, s=18)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set(xlabel="predicted", ylabel="residual",
            title="Residuals vs. fitted — note the funnel")

lo, hi = min(y.min(), y_hat.min()), max(y.max(), y_hat.max())
axes[1].scatter(y_test, y_hat, alpha=0.6, s=18)
axes[1].plot([lo, hi], [lo, hi], "k--", lw=0.8)
axes[1].set(xlabel="actual", ylabel="predicted",
            title=f"Predicted vs. actual  RMSE={mean_squared_error(y_test, y_hat) ** 0.5:.1f}")
plt.tight_layout()
plt.show()


The funnel in the left plot tells you the noise grows with the magnitude of $y$.
A constant RMSE summary hides this; the chart makes it obvious in a second. **If the
practical use of the model involves reporting uncertainty, you must either explicitly
model heteroskedasticity (e.g. quantile regression) or change your loss.**

## 4. Discrimination vs. calibration

A model can be **good at ranking** (high AUC) but **badly calibrated** (probabilities
are systematically off). And vice versa.

- Random forests are often **under-confident** — they squash probabilities toward 0.5.
- SVMs with Platt scaling are usually well calibrated.
- Deep nets with cross-entropy are usually **over-confident** out of the box, especially
  on out-of-distribution inputs.

If you only ever need the **top-K predictions**, discrimination is what matters. If you
will **act on the probability** — e.g. routing only "high-risk" cases for human review —
calibration matters at least as much. The fix is post-hoc: **Platt scaling** (logistic
regression on the model's logits) or **isotonic regression** (monotonic regression on
the predicted probabilities).

## Summary

- The **chart matches the question**: ROC for ranking, PR for rare positives,
  calibration for probability-as-truth.
- **Decision boundaries** make biases visible. Use them on 2D toys for intuition and on
  2D projections for real classifiers.
- **Residual plots** are mandatory for regression. RMSE is a number; the residual plot
  is what diagnoses the model.
- **Discrimination and calibration are different problems.** Both can be measured, both
  can be charted. Report both when the downstream task uses the probability.

In the lab we build a full diagnostic suite for one classifier and one regressor, mostly
with Yellowbrick and scikit-plot, falling back to custom Matplotlib where they don't
cover the case.
